In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer


df = pd.read_csv('car_purchasing.csv', encoding='ISO-8859-1')

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   customer name        500 non-null    object 
 1   JobTitle             500 non-null    object 
 2   customer e-mail      500 non-null    object 
 3   country              500 non-null    object 
 4   gender               500 non-null    int64  
 5   age                  500 non-null    int64  
 6   BasePay              500 non-null    float64
 7   OvertimePay          500 non-null    float64
 8   OtherPay             500 non-null    float64
 9   Benefits             0 non-null      float64
 10  TotalPay             500 non-null    float64
 11  TotalPayBenefits     500 non-null    float64
 12  credit card debt     500 non-null    float64
 13  net worth            500 non-null    float64
 14  car purchase amount  500 non-null    float64
dtypes: float64(9), int64(2), object(4)
memor

In [3]:
df.drop(['Benefits', 'customer e-mail'], axis=1, inplace=True)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   customer name        500 non-null    object 
 1   JobTitle             500 non-null    object 
 2   country              500 non-null    object 
 3   gender               500 non-null    int64  
 4   age                  500 non-null    int64  
 5   BasePay              500 non-null    float64
 6   OvertimePay          500 non-null    float64
 7   OtherPay             500 non-null    float64
 8   TotalPay             500 non-null    float64
 9   TotalPayBenefits     500 non-null    float64
 10  credit card debt     500 non-null    float64
 11  net worth            500 non-null    float64
 12  car purchase amount  500 non-null    float64
dtypes: float64(8), int64(2), object(3)
memory usage: 50.9+ KB


In [5]:
x = df.drop('car purchase amount', axis=1)
y = df['car purchase amount']

In [6]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [7]:
numerical_features = x_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = x_train.select_dtypes(include=['object']).columns

In [8]:
print(numerical_features)
print(categorical_features)

Index(['gender', 'age', 'BasePay', 'OvertimePay', 'OtherPay', 'TotalPay',
       'TotalPayBenefits', 'credit card debt', 'net worth'],
      dtype='object')
Index(['customer name', 'JobTitle', 'country'], dtype='object')


In [9]:
sc = StandardScaler()
oe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
num_transformer = Pipeline(
    steps=[
      ('imputer', SimpleImputer(strategy='mean')),
      ('scaler', sc)
     ]
    )

cat_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', oe)
    ]
    )

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, numerical_features),
        ('cat', cat_transformer, categorical_features)
    ]
)

In [11]:
x_train_preprocessed = preprocessor.fit_transform(x_train)
y_train_preprocessed = np.log1p(y_train)

In [12]:
x_test_preprocessed = preprocessor.transform(x_test)
y_test_preprocessed = np.log1p(y_test.values)

c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [13]:
x_train_tensor = torch.FloatTensor(x_train_preprocessed)
x_test_tensor = torch.FloatTensor(x_test_preprocessed)
y_train_tensor = torch.FloatTensor(y_train_preprocessed.values).reshape(-1, 1)
y_test_tensor = torch.FloatTensor(y_test_preprocessed).reshape(-1, 1)

In [14]:

print("\n✅ Tensors created from PREPROCESSED data!")
print(f"x_train_tensor: {x_train_tensor.shape}")
print(f"x_test_tensor: {x_test_tensor.shape}")
print(f"y_train_tensor: {y_train_tensor.shape}")
print(f"y_test_tensor: {y_test_tensor.shape}")


✅ Tensors created from PREPROCESSED data!
x_train_tensor: torch.Size([400, 668])
x_test_tensor: torch.Size([100, 668])
y_train_tensor: torch.Size([400, 1])
y_test_tensor: torch.Size([100, 1])


In [15]:
class HomePrices(nn.Module):
    def __init__(self, input_features):
        super(HomePrices, self).__init__()
        self.layer_1 = nn.Linear(input_size, 32)
        self.layer_2 = nn.Linear(32, 16)
        self.layer_3 = nn.Linear(16, 8)
        self.layer_4 = nn.Linear(8, 4)
        self.layer_5 = nn.Linear(4, 2)
        self.layer_6 = nn.Linear(2, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = torch.relu(self.layer_1(x))
        x = self.dropout(x)
        x = torch.relu(self.layer_2(x))
        x = self.dropout(x)
        x = torch.relu(self.layer_3(x))
        x = self.dropout(x)
        x = torch.relu(self.layer_4(x))
        x = self.dropout(x)
        x = torch.relu(self.layer_5(x))
        x = self.dropout(x)
        x = self.relu(self.layer_6(x))
        return x

In [16]:
input_size = x_train_tensor.shape[1]
model = HomePrices(input_size)

In [17]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [18]:
num_epoch = 200
batch_size = 64
train_loss = []
test_losses = []


In [19]:
for i in range(num_epoch):
    model.train()
    epoch_loss = 0
    for j in range(0, len(x_train_tensor), batch_size):
        batch_x = x_train_tensor[j:j+batch_size]
        batch_y = y_train_tensor[j:j+batch_size]
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_train_loss = epoch_loss / (len(x_train_tensor) / batch_size)

    with torch.no_grad():
        test_outputs = model(x_test_tensor)
        test_loss = criterion(test_outputs, y_test_tensor)
        test_losses.append(test_loss.item())

        if (i+1) % 20 == 0:
            print(f"Epoch {i+1} / {num_epoch} Train loss: {avg_train_loss} Test loss: {test_loss.item()}")

    model.eval()
    with torch.no_grad():
        train_outputs = model(x_train_tensor)
        train_loss = criterion(train_outputs, y_train_tensor)
        train_mse = mean_squared_error(y_train_tensor.numpy(), train_outputs.numpy())
        train_rmse = np.sqrt(train_mse)

        test_outputs = model(x_test_tensor)
        test_loss = criterion(test_outputs, y_test_tensor)
        test_mse = mean_squared_error(y_test_tensor.numpy(), test_outputs.numpy())
        test_rmse = np.sqrt(test_mse)

        print("training loss:", train_loss.item())
        print("training MSE:", train_mse)
        print("training RMSE:", train_rmse)
        print("test loss:", test_loss.item())
        print("test MSE:", test_mse)
        print("test RMSE:", test_rmse)

training loss: 104.20368194580078
training MSE: 104.20368957519531
training RMSE: 10.208020845158737
test loss: 104.90254211425781
test MSE: 104.90254211425781
test RMSE: 10.242194204088195
training loss: 104.18255615234375
training MSE: 104.18254852294922
training RMSE: 10.20698528082358
test loss: 104.88134002685547
test MSE: 104.8813247680664
test RMSE: 10.241158370422088
training loss: 104.16145324707031
training MSE: 104.16144561767578
training RMSE: 10.205951480272468
test loss: 104.86016845703125
test MSE: 104.86015319824219
test RMSE: 10.240124667124038
training loss: 104.14026641845703
training MSE: 104.14025115966797
training RMSE: 10.204913089275575
test loss: 104.8389663696289
test MSE: 104.83897399902344
test RMSE: 10.239090486904754
training loss: 104.11851501464844
training MSE: 104.11851501464844
training RMSE: 10.203848049370809
test loss: 104.8174819946289
test MSE: 104.81748962402344
test RMSE: 10.238041298218299
training loss: 104.0951919555664
training MSE: 104.095